## **Connecting ARGUS to the Sage Continuum sensor-network MCP server**

This notebook demonstrates **ARGUS** (Agentic Researcher Guide for Unified Science) extended via **MCP (Model Context Protocol)**. The `%%mcp` cell
below registers the [Sage Continuum](https://sagecontinuum.org) MCP server — an NSF cyberinfrastructure platform that operates a distributed network
of hundreds of edge-computing sensor nodes deployed across the US and internationally. Each analytical step is then expressed as a plain English
`%%ask` prompt.

**What this notebook does:**
1. Lists all available Sage Continuum nodes and their current status
2. Inspects details about the BME680 environmental sensor (temperature, humidity, pressure, gas)
3. Reads current temperature readings from a specific node (W023)
4. Retrieves recent camera images from another node (W019)



####   **Choose your language model**
  
  Argus is model-agnostic — you can plug in any of the major LLM providers and Argus will work. Pick whichever you have an API key for. To switch, comment out the active `LLM = {...}` block and uncomment the one you want.

  **This notebook defaults to GLM-5 via the National Research Platform** (free for NRP users, strong on scientific tool use). If you don't have NRP
  access, swap in OpenAI, Anthropic, or Google — they all work.

  **Before you run the cell: save your API key as a Colab Secret**

  Argus reads your API key from a Colab Secret, not from the notebook itself — that keeps the key out of your saved file and out of anyone you share the
  notebook with.

  1. Click the 🔑 key icon in the left sidebar (**Secrets**).
  2. Click **+ Add new secret**.
  3. Set the **Name** to match the `api_key_env` value in your `LLM` block:
     - `NRP_API_KEY` for GLM-5 on NRP
     - `OPENAI_API_KEY` for GPT-5.5
     - `ANTHROPIC_API_KEY` for Claude Sonnet/Opus
     - `GOOGLE_API_KEY` for Gemini

  4. Paste your key into the **Value** field.
  5. Toggle **Notebook access** on for this notebook.

  If the key isn't set (or notebook access is off), Argus will fail with an authentication error when it tries to call the model.

  ### Supported models

  | Provider | Models |
  |---|---|
  | OpenAI | GPT-5.5, GPT-5.4 |
  | Anthropic | Claude Opus 4.7, Opus 4.6 |
  | Google | Gemini 3.5 Flash |
  | Z.ai | GLM-5, GLM-5.1 |
  | Moonshot | Kimi-K2.6 |
  | MiniMax | MiniMax-M2.7 |
  | DeepSeek | DeepSeek-V4 Flash |

  Any model in this list should run Argus notebooks correctly. The four pre-configured blocks below are starter templates; for the others, follow the
  same `LLM = {...}` pattern with the right `model` and `api_key_env`.

In [1]:
  # Use GLM-5 from National Research Platform
  LLM = {
      "model": "glm-5",
      "url": "https://ellm.nrp-nautilus.io/v1",
      "api_key_env": "NRP_API_KEY",
  }

  # Use GPT-5.5 from OpenAI
  # LLM = {
  #     "model": "gpt-5.5",
  #     "api_key_env": "OPENAI_API_KEY",
  # }

  # Use Claude Sonnet 4.6 from Anthropic
  # LLM = {
  #     "model": "claude-sonnet-4-6",
  #     "api_key_env": "ANTHROPIC_API_KEY",
  #     "flavor": "anthropic",
  # }

  # Use Gemini 3.5 Flash from Google
  # LLM = {
  #     "model": "gemini-3.5-flash",
  #     "api_key_env": "GOOGLE_API_KEY",
  #     "flavor": "gemini",
  # }


#### **Install Argus from GitHub Repo**

In [2]:
import urllib.request
exec(urllib.request.urlopen(
    'https://raw.githubusercontent.com/klinucsd/sage/main/argus_colab/install.py'
).read().decode(), globals())

ARGUS bootstrap starting...
  ✓ Wrote LLM config to /root/.deepagents/config.toml (added default params for 1 ChatOpenAI provider(s))
  ✓ Loaded NRP_API_KEY from Colab Secrets
  · Installing ARGUS dependencies (~30–90 s the first time)...
  ✓ ARGUS dependencies installed
  ✓ Downloaded sage_magic.py + sage_kernel_backend.py from main
  ✓ Installed 7 core skill(s) into /root/.deepagents/agent/skills: ndp-search, sage-bbox-map, sage-dropdown, sage-metrics, skillsmp, us-counties, us-states

ARGUS ready — %%ask, %%mcp, %%skill are registered.
  Provider: llm  ·  Model: llm:glm-5


In [3]:
%reset

**Sage reset.** Output folder cleared, history cleared.

_To also clear cell outputs: click **Edit → Clear all outputs** (in the Colab menu bar)._

#### **Connect to the Sage Continuum sensor network**

This notebook talks to the Sage Continuum MCP server, which requires your Sage Continuum credentials. ARGUS reads them from Colab Secrets — same
pattern as your language-model API key. They never appear in the notebook itself.

Add these two secrets:

| Secret name | Value |
|---|---|
| `SAGECONTINUUM_USERNAME` | Your Sage Continuum username |
| `SAGECONTINUUM_TOKEN` | Your Sage Continuum API token (from [portal.sagecontinuum.org](https://portal.sagecontinuum.org) → Account → API Tokens) |

**To add a secret:**
1. Click the 🔑 key icon in the left sidebar (**Secrets**)
2. **+ Add new secret**, set the name exactly as shown above, paste the value
3. Toggle **Notebook access** on

If a secret is missing when the next cell runs, ARGUS will fail with a clear message telling you which one to add.

In [4]:
%%mcp
  {
    "mcpServers": {
      "sage_continuum": {
        "url": "https://mcp.sagecontinuum.org/mcp",
        "headers": {
          "Authorization": "Bearer $SAGECONTINUUM_USERNAME:$SAGECONTINUUM_TOKEN"
        }
      }
    }
  }

In [5]:
%%ask
What's the highest temperature recorded today across all nodes?

Let me try with a standard time range format.

The highest temperature recorded across all Sage nodes in the last 24 hours is **34.20°C** (~93.6°F).

This comes from the environment (bme680) sensor network, which logged 1,000 readings from 33 unique sensors. The overall average was -5.85°C, though that's pulled down by some apparent sensor errors (the -132.85°C minimum is clearly an outlier/malfunction rather than a real reading).

In [6]:
%%ask
Show me details about BME680 environmental sensors

Here are the details for the **BME680** environmental sensor:

| Field | Details |
|---|---|
| **Hardware ID** | bme680 |
| **Manufacturer** | Bosch |
| **Capabilities** | Temperature, Pressure, Humidity, Gas |
| **Datasheet** | [Bosch BME680 Datasheet (PDF)](https://www.bosch-sensortec.com/media/boschsensortec/downloads/datasheets/bst-bme680-ds001.pdf) |

**Description:** The BME680 is the first gas sensor that integrates high-linearity and high-accuracy gas, pressure, humidity, and temperature sensors. It is especially developed for mobile applications and wearables where size and low power consumption are critical.

**Measured quantities:**
- 🌡️ **Temperature** — ambient air temperature
- 💧 **Humidity** — relative humidity
- 🌀 **Pressure** — barometric pressure
- 🧪 **Gas** — volatile organic compound (VOC) / air quality index

**Deployment:** The BME680 is the most widely deployed sensor in the Sage network, present in **~130 nodes** across the fleet (including W-series weather nodes, V-series nodes, and T002).

In [7]:
%%ask
Show me current temperature readings from node W023

Here are the current temperature readings for **node W023** :

| Sensor | Type | Latest Reading | Avg (Last Hour) | Min / Max | Status |
|---|---|---|---|---|---|
| **BME680** | Environment | -131.77°C | -130.26°C | -131.77°C / 32.99°C | ⚠️ Likely malfunctioning |
| **BME280** | Internal/Hardware | 42.98°C | 42.98°C | 42.98°C / 42.98°C | ✅ Normal |

**Key observations:**

- The **BME680** (environmental) sensor is clearly malfunctioning — a reading of -131.77°C is physically impossible and indicates a hardware fault or communication error. Earlier in the hour it was reading normally (up to 32.99°C), suggesting the sensor went offline or corrupted mid-session.

- The **BME280** (internal/hardware) sensor reports 42.98°C, which is a reasonable board-level temperature for an outdoor node in warm conditions. However, it only has one reading in the last hour, so it may be sampling infrequently.

- The BME680's max of 32.99°C from the past hour is likely the last valid environmental temperature before the sensor faulted.

In [8]:
%%ask
Find recent camera images from node W019

Node **W019** has recent camera images from both its top and bottom cameras. Here's a summary:

### Recent Camera Images from W019

| Camera | Timestamp (UTC) | Image URL |
|---|---|---|
| **Top Camera** | 2026-06-14 18:00:13 | [View Image](https://storage.sagecontinuum.org/api/v1/data/imagesampler-top-10/sage-imagesampler-top-0.3.5/000048b02d15bc87/1781460013678891266-top_camera.jpg) |
| **Bottom Camera** | 2026-06-14 18:00:13 | [View Image](https://storage.sagecontinuum.org/api/v1/data/imagesampler-bottom-10/sage-imagesampler-bottom-0.3.5/000048b02d15bc87/1781460013382399031-bottom_camera.jpg) |

### Additional Image-Related Data (Last Hour)

| Measurement | Latest Value | Description |
|---|---|---|
| **Cloud Coverage** | 1.00 (100%) | Fully overcast skies |
| **Motion Detected** | 5 events | Cloud motion detected |
| **Motion Direction** | 185° (S) | Clouds moving southward |
| **Motion Magnitude** | 0.00 px/min | Minimal cloud speed |

The node is running the **imagesampler** plugin (v0.3.5) on both cameras, along with a **cloud cover** analysis plugin. The cloud coverage reading of 1.0 indicates completely overcast conditions at this node's location.

In [9]:
%%ask
List all available Sage nodes and their current status

Here's a summary of all **287 Sage nodes** and their current status:

### Deployed & Active Nodes: 155

| Series | Count | Key Locations |
|---|---|---|
| **D** | 1 | — |
| **H** | 20 | Chicago IL, Lemont IL, Eugene OR, Nunn CO |
| **N** | 2 | Chicago IL, Lemont IL |
| **RH** | 1 | — |
| **T** | 1 | — |
| **V** | 38 | AZ, AK, CA, CO, IL, MA, MI, NM, ND, OK, OR, PR, UT, VA, UK |
| **W** | 89 | Chicago IL (many), Argonne IL, Austin TX, Eugene OR, Fairbanks AK, Fresno CA, Lahaina HI, Lubbock TX, Markham IL, Moran WY, Richland WA, Salt Lake City UT, Sterling CO, Villa Park IL, UK, and more |
| **X** | 1 | Weld County, CO |
| **WT** | 1 | — |

### Other Nodes (Not Currently Deployed): 132

| Status | Count | Description |
|---|---|---|
| **Awaiting Deployment** | 38 | Hardware ready, not yet installed |
| **Shipment Pending** | 22 | In transit to deployment site |
| **Maintenance** | 16 | Offline for repairs/updates |
| **Standby** | 5 | Powered down but available |
| **Retired** | 4 | Permanently decommissioned (V003, V009, V012, W056) |
| **None / Unknown** | 47 | No status recorded |

### Geographic Highlights
- **Chicago metro area** has the densest concentration (~40+ W-series nodes)
- **Argonne National Laboratory** (Lemont, IL) hosts many V and W nodes
- **West Coast** nodes span from Alaska to Southern California
- **International** nodes exist in the **United Kingdom** (W06B Plockton, W0BF Birmingham) and **Puerto Rico** (V055, V056)
- Nodes are deployed in diverse environments: urban streets, nature preserves, mountaintops (Cuyamaca Peak, Palomar Mountain), volcanic parks (Hawaii Volcanoes NP), and remote tundra (Healy AK, Northwest Stutsman ND)